# 06 - Model Comparison, Validation Gap Analysis, and Error Diagnostics

### Research Question
How do prediction error estimates (RMSE, MAE, R², RMSLE) obtained with random k-fold cross-validation differ from those obtained with chronological validation, in which direction and by how much, and does this difference depend on the model architecture (linear vs. tree ensembles)?

### Structure of this Notebook
1. **Fold-averaged summary**: benchmarks of all models and schemes, as saved by the model notebooks.
2. **Matched 2012 evaluation**: all schemes scored on the identical hours that the chronological scheme tests.
3. **Validation gap**: absolute and relative RMSE increase from random to chronological validation, with block-bootstrap uncertainty.
4. **Diagnostics**: error by test period, residuals against predictions, bias over time, peak versus off-peak hours.
5. **Conclusions and limitations.**

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.data import load_data
from src.validation import RESULTS_DIR, SCHEMES, SEED, compute_metrics

plt.style.use("default")               # reset any inherited theme (e.g. a dark one)
Path("figures").mkdir(exist_ok=True)   # figures are saved here

## 0. Loading the Results

Every model notebook saves `results/{model}_{scheme}_folds.csv` and `results/{model}_{scheme}_preds.csv`. This notebook trains nothing; it only reads them. If any file is missing, loading stops with the list of missing files instead of silently leaving a model out of the figures.

In [ ]:
MODELS = ["baseline_const", "baseline_hour_day", "ridge", "ridge_log", "rf", "hgb", "hgb_poisson"]
NAMES = {
    "baseline_const": "Baseline: constant", "baseline_hour_day": "Baseline: hour × day type",
    "ridge": "Ridge", "ridge_log": "Ridge (log target)", "rf": "Random forest",
    "hgb": "Boosting (squared error)", "hgb_poisson": "Boosting (Poisson)",
}

missing = [f"{m}_{s}_{kind}.csv" for m in MODELS for s in SCHEMES for kind in ("folds", "preds")
           if not (RESULTS_DIR / f"{m}_{s}_{kind}.csv").exists()]
if missing:
    raise FileNotFoundError("Missing result files (run the model notebooks or git pull):\n" + "\n".join(missing))

folds = {(m, s): pd.read_csv(RESULTS_DIR / f"{m}_{s}_folds.csv", parse_dates=["test_start", "test_end"])
         for m in MODELS for s in SCHEMES}
preds = {(m, s): pd.read_csv(RESULTS_DIR / f"{m}_{s}_preds.csv", index_col="datetime", parse_dates=True)
         for m in MODELS for s in SCHEMES}

print(f"Loaded {len(MODELS)} models x {len(SCHEMES)} schemes")

## 1. Fold-Averaged Summary

*Random and blocked validation test on all hours of 2011-2012, whereas chronological validation tests only on 2012, so these fold averages compare models within one scheme; they do not measure the validation gap (see Section 2).*

In [ ]:
metric_cols = ["rmse_mean", "rmse_std", "mae_mean", "r2_mean"]
summary = pd.DataFrame([{
    "model": m, "scheme": s,
    "rmse_mean": folds[(m, s)]["rmse"].mean(), "rmse_std": folds[(m, s)]["rmse"].std(),
    "mae_mean": folds[(m, s)]["mae"].mean(), "r2_mean": folds[(m, s)]["r2"].mean(),
} for m in MODELS for s in SCHEMES])

summary_wide = (summary.pivot(index="model", columns="scheme")[metric_cols]
                .reindex(columns=pd.MultiIndex.from_product([metric_cols, SCHEMES]))
                .reindex(MODELS))
summary_wide.round(2)

## 2. Matched Evaluation on Identical 2012 Hours

The chronological scheme tests only on 2012. To compare schemes fairly, every model's out-of-fold predictions are restricted to exactly those hours, so any difference between schemes comes from the scheme and not from the test period. The check below confirms that all models have the same chronological test hours.

`bias` is the mean residual (actual - predicted): positive means the model underestimates demand.

In [ ]:
chrono_index = preds[(MODELS[0], "chrono")].index
for m in MODELS:
    assert preds[(m, "chrono")].index.equals(chrono_index), f"{m}: different chronological test hours"

matched = {key: p.loc[chrono_index] for key, p in preds.items()}
print(f"Matched set: {len(chrono_index):,} hours, {chrono_index.min():%Y-%m-%d} to {chrono_index.max():%Y-%m-%d}")

rows = []
for (m, s), p in matched.items():
    rows.append({"model": m, "scheme": s, "bias": (p["y_true"] - p["y_pred"]).mean(),
                 **compute_metrics(p["y_true"], p["y_pred"])})
df_matched = pd.DataFrame(rows)


def wide(metric):
    return df_matched.pivot(index="model", columns="scheme", values=metric)[list(SCHEMES)].reindex(MODELS)


rmse_matched, mae_matched, bias_matched = wide("rmse"), wide("mae"), wide("bias")

print("\nRMSE on matched hours")
display(rmse_matched.round(1))
print("MAE on matched hours")
display(mae_matched.round(1))
print("Bias (mean residual) on matched hours")
display(bias_matched.round(1))

## 3. The Validation Gap

$$\Delta \text{RMSE} = \text{RMSE}_{\text{chrono}} - \text{RMSE}_{\text{random}}, \qquad \% \text{ gap} = \frac{\Delta \text{RMSE}}{\text{RMSE}_{\text{random}}} \times 100$$

A single RMSE per cell hides how much it would change with a different sample of test hours. Neighbouring hours are strongly correlated, so ordinary confidence intervals would be too narrow. We therefore use a **block bootstrap over calendar weeks**: whole weeks of the matched set are resampled with replacement, and the same resampled weeks are used for every model and scheme (so comparisons between them are paired). The intervals describe the uncertainty from the choice of test hours only; they do not include the randomness of training or of the hyperparameter search.

In [ ]:
N_BOOT = 2000
week_id = pd.factorize(chrono_index.to_period("W"))[0]
n_weeks = week_id.max() + 1
week_counts = np.bincount(week_id, minlength=n_weeks)
draw = np.random.default_rng(SEED).integers(0, n_weeks, size=(N_BOOT, n_weeks))  # same weeks for all models

boot_rmse = {}
for key, p in matched.items():
    week_sq_err = np.bincount(week_id, weights=((p["y_true"] - p["y_pred"]) ** 2).to_numpy(), minlength=n_weeks)
    boot_rmse[key] = np.sqrt(week_sq_err[draw].sum(axis=1) / week_counts[draw].sum(axis=1))


def ci(values):
    return np.percentile(values, [2.5, 97.5])


gap = pd.DataFrame({
    "random": rmse_matched["random"], "blocked": rmse_matched["blocked"], "chrono": rmse_matched["chrono"],
})
gap["delta"] = gap["chrono"] - gap["random"]
gap["gap_pct"] = gap["delta"] / gap["random"] * 100
boot_gap = {m: (boot_rmse[(m, "chrono")] / boot_rmse[(m, "random")] - 1) * 100 for m in MODELS}
gap["gap_pct_low"] = [ci(boot_gap[m])[0] for m in MODELS]
gap["gap_pct_high"] = [ci(boot_gap[m])[1] for m in MODELS]
display(gap.round(1))

# improvement over the hour x day type baseline, per scheme
gain = (1 - rmse_matched.div(rmse_matched.loc["baseline_hour_day"])) * 100
print("RMSE improvement over the hour x day type baseline (%)")
display(gain.round(1))

In [ ]:
# colour = validation scheme (random is the optimistic one, hence the warm colour)
colors = {"random": "#E8A33D", "blocked": "#5BAFC9", "chrono": "#26547C"}
scheme_names = {"random": "Random", "blocked": "Blocked", "chrono": "Chronological"}

order = rmse_matched["chrono"].sort_values().index.tolist()      # best chronological model at the top
ypos = {m: len(order) - 1 - i for i, m in enumerate(order)}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.8), dpi=110, sharey=True,
                               gridspec_kw={"width_ratios": [1.7, 1]})

for m in order:
    y0 = ypos[m]
    values = [rmse_matched.loc[m, s] for s in SCHEMES]
    ax1.plot([min(values), max(values)], [y0, y0], color="#CDD2D8", lw=4, solid_capstyle="round", zorder=1)
    for s in SCHEMES:
        low, high = np.percentile(boot_rmse[(m, s)], [2.5, 97.5])
        ax1.hlines(y0, low, high, color=colors[s], lw=1.6, zorder=2)
        ax1.scatter(rmse_matched.loc[m, s], y0, s=70, color=colors[s], edgecolor="white", linewidth=1.2, zorder=3)

for s in SCHEMES:                                                 # legend entries
    ax1.scatter([], [], s=70, color=colors[s], label=scheme_names[s])
ax1.legend(loc="upper center", bbox_to_anchor=(0.5, -0.14), ncol=3, frameon=False, fontsize=9)
ax1.set_yticks(list(ypos.values()))
ax1.set_yticklabels([NAMES[m] for m in ypos])
ax1.set_xlabel("RMSE (rentals / hour)")
ax1.set_title("RMSE on identical 2012 hours", loc="left", fontweight="bold", fontsize=12)

g = gap.loc[order]
y_pos = [ypos[m] for m in order]
ax2.barh(y_pos, g["gap_pct"], color="#7D8FA3", height=0.55, zorder=2)
ax2.errorbar(g["gap_pct"], y_pos, xerr=[g["gap_pct"] - g["gap_pct_low"], g["gap_pct_high"] - g["gap_pct"]],
             fmt="none", ecolor="#2B2F36", elinewidth=1.2, capsize=3, zorder=3)
for y0, val, high in zip(y_pos, g["gap_pct"], g["gap_pct_high"]):
    ax2.text(high + 1.5, y0, f"{val:+.0f}%", va="center", fontsize=9, color="#2B2F36")
ax2.set_xlabel("RMSE increase, chronological vs random (%)")
ax2.set_xlim(0, g["gap_pct_high"].max() * 1.2)
ax2.set_title("Optimism of random CV", loc="left", fontweight="bold", fontsize=12)

for ax in (ax1, ax2):
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    ax.set_axisbelow(True)
    ax.grid(axis="x", color="#E6E9ED")

fig.text(0.01, -0.02, "Whiskers: 95% block bootstrap over weeks.", fontsize=8.5, color="#8A9099")
plt.tight_layout()
fig.savefig("figures/comparison_matched_rmse.png", dpi=200, bbox_inches="tight")
plt.show()

## 4. Error and Residual Diagnostics

All diagnostics use the chronological predictions on the matched 2012 hours, the setting that resembles real forecasting.

### 4.1 Error by test period

Each chronological fold is a different part of 2012 predicted by a model trained on everything before it. The plot shows whether the error is stable across periods or concentrated in some of them (the constant baseline is left out because its scale compresses the rest).

In [ ]:
# colour = model (greys for baselines, purple Ridge, green forest, red boosting)
colors = {"baseline_hour_day": "#7D848E", "ridge": "#8E6BBF", "ridge_log": "#C3A8E0",
          "rf": "#2A9D8F", "hgb": "#D1495B", "hgb_poisson": "#F2A5AE"}

f0 = folds[(MODELS[0], "chrono")]
xt_labels = [f"{a:%b}-{b:%b}\ntrain n={n:,}" for a, b, n in zip(f0["test_start"], f0["test_end"], f0["n_train"])]

fig, ax = plt.subplots(figsize=(9.5, 4.8), dpi=110)
for m, color in colors.items():
    is_base = m.startswith("baseline")
    ax.plot(folds[(m, "chrono")]["fold"], folds[(m, "chrono")]["rmse"], marker="o", color=color,
            ls="--" if is_base else "-", lw=1.6 if is_base else 2.2, label=NAMES[m])

ax.set_xticks(f0["fold"])
ax.set_xticklabels(xt_labels)
ax.set_xlabel("Test period of the chronological fold (2012)")
ax.set_ylabel("RMSE (rentals / hour)")
ax.set_title("Chronological RMSE by test period", loc="left", fontweight="bold", fontsize=12)
ax.legend(loc="center left", bbox_to_anchor=(1.01, 0.5), frameon=False, fontsize=9)
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
ax.set_axisbelow(True)
ax.grid(color="#E6E9ED")
plt.tight_layout()
fig.savefig("figures/chrono_rmse_by_fold.png", dpi=200, bbox_inches="tight")
plt.show()

### 4.2 Residuals against predictions

Each panel is a density of hours (darker = more hours, log scale); the red line is the mean residual within bins of the predicted value, so it shows whether the model is biased at low or at high predictions. Binning by the *prediction* is the right way to check calibration. (Binning by the *actual* value would mechanically produce positive residuals at high demand and negative at low demand for any noisy model, so it cannot show a model-specific failure.)

In [ ]:
DIAG = ["baseline_hour_day", "ridge", "ridge_log", "rf", "hgb", "hgb_poisson"]
fig, axes = plt.subplots(2, 3, figsize=(13, 7.2), dpi=110, sharex=True, sharey=True, constrained_layout=True)

for ax, m in zip(axes.ravel(), DIAG):
    p = matched[(m, "chrono")]
    res = p["y_true"] - p["y_pred"]
    hb = ax.hexbin(p["y_pred"], res, gridsize=36, mincnt=1, bins="log", cmap="Blues", linewidths=0,
                   vmin=1, vmax=150)
    binned = pd.DataFrame({"x": p["y_pred"], "r": res}).groupby(
        pd.qcut(p["y_pred"], 12, duplicates="drop"), observed=True).mean()
    ax.plot(binned["x"], binned["r"], color="#C8553D", lw=2.2)          # mean residual per prediction bin
    ax.axhline(0, color="#2B2F36", lw=1, ls="--")
    ax.set_title(NAMES[m], loc="left", fontweight="bold", fontsize=12)
    ax.text(0.97, 0.05, f"mean residual {res.mean():+.0f}", transform=ax.transAxes, ha="right",
            fontsize=9, color="#2B2F36")
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    ax.set_axisbelow(True)
    ax.grid(color="#E6E9ED")

fig.supxlabel("Predicted demand (rentals / hour)")
fig.supylabel("Residual (actual - predicted)")
fig.colorbar(hb, ax=axes, shrink=0.6, pad=0.02, label="hours (log scale)")
fig.savefig("figures/residuals_vs_predicted.png", dpi=200, bbox_inches="tight")
plt.show()

### 4.3 Bias over time

If every model underestimates every month of 2012 by a similar amount, the cause is a shared one (here, the growth in demand between the years, which none of the feature sets represents) rather than a property of one model family.

In [ ]:
colors = {"baseline_hour_day": "#7D848E", "ridge": "#8E6BBF", "ridge_log": "#C3A8E0",
          "rf": "#2A9D8F", "hgb": "#D1495B", "hgb_poisson": "#F2A5AE"}

fig, ax = plt.subplots(figsize=(9.5, 4.6), dpi=110)
for m, color in colors.items():
    p = matched[(m, "chrono")]
    monthly = (p["y_true"] - p["y_pred"]).resample("MS").mean()
    is_base = m.startswith("baseline")
    ax.plot(monthly.index.month, monthly.values, marker="o", color=color, ls="--" if is_base else "-",
            lw=1.6 if is_base else 2.2, label=NAMES[m])

ax.axhline(0, color="#2B2F36", lw=1)
ax.set_xticks(range(1, 13))
ax.set_xticklabels(["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"])
ax.set_ylabel("Mean residual (actual - predicted)")
ax.set_title("Bias by month, 2012 (chronological)", loc="left", fontweight="bold", fontsize=12)
ax.legend(loc="center left", bbox_to_anchor=(1.01, 0.5), frameon=False, fontsize=9)
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
ax.set_axisbelow(True)
ax.grid(color="#E6E9ED")
plt.tight_layout()
fig.savefig("figures/bias_by_month.png", dpi=200, bbox_inches="tight")
plt.show()

### 4.4 Peak versus off-peak hours

Peak hours are working-day commuting hours (07-09 and 17-19), where demand is highest. Splitting by hour and day type uses information the model also has, so it does not suffer from the selection problem of splitting by the actual value.

In [ ]:
X_all, _ = load_data()
feat = X_all.loc[chrono_index, ["hr", "workingday"]]
is_peak = (feat["hr"].isin([7, 8, 9, 17, 18, 19]) & (feat["workingday"] == 1)).to_numpy()
print(f"Peak hours: {is_peak.sum():,} of {len(is_peak):,} ({is_peak.mean():.0%})")

rows = []
for m in MODELS[1:]:
    p = matched[(m, "chrono")]
    res = (p["y_true"] - p["y_pred"]).to_numpy()
    for period, mask in [("Off-peak", ~is_peak), ("Peak", is_peak)]:
        rows.append({"model": m, "period": period, "rmse": np.sqrt((res[mask] ** 2).mean()), "bias": res[mask].mean()})
peak = pd.DataFrame(rows)
display(peak.pivot(index="model", columns="period", values=["rmse", "bias"]).reindex(MODELS[1:]).round(1))

# colour = period (grey off-peak, accent peak)
period_colors = {"Off-peak": "#B9BEC5", "Peak": "#C8553D"}

fig, axes = plt.subplots(1, 2, figsize=(12, 4.6), dpi=110, sharey=True)
y_idx = np.arange(len(MODELS[1:]))[::-1]
for ax, metric, title in zip(axes, ["rmse", "bias"], ["RMSE", "Bias (mean residual)"]):
    for offset, period in zip([0.19, -0.19], ["Off-peak", "Peak"]):
        vals = peak[peak["period"] == period].set_index("model").loc[MODELS[1:], metric]
        ax.barh(y_idx + offset, vals.values, height=0.36, color=period_colors[period], label=period, zorder=2)
    ax.set_title(title, loc="left", fontweight="bold", fontsize=12)
    ax.set_xlabel("rentals / hour")
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    ax.set_axisbelow(True)
    ax.grid(axis="x", color="#E6E9ED")

axes[0].set_yticks(y_idx)
axes[0].set_yticklabels([NAMES[m] for m in MODELS[1:]])
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=2, frameon=False, fontsize=9, bbox_to_anchor=(0.5, -0.04))
plt.tight_layout()
fig.savefig("figures/peak_vs_offpeak.png", dpi=200, bbox_inches="tight")
plt.show()

## 5. Conclusions and Limitations

*Fill in the numbers from the tables above after the final run; the statements below are limited to what Sections 2-4 can support.*

### 1. Direction and size of the validation gap
- On identical 2012 hours, random cross-validation gives the lowest error for every model, and chronological validation the highest or close to it. The size of the gap is in the table of Section 3 with its bootstrap interval: +34% for Ridge, +37% for Ridge with a log target, +65% for the random forest, boosting: **[fill in]**.
- The most likely mechanism is that shuffling puts neighbouring, strongly correlated hours into both the training and the test part. The experiment does not test this mechanism directly; the evidence for it is that blocked validation, which removes this overlap, already lands close to chronological validation.

### 2. Dependence on the model
- The gap grows with model flexibility: it is smallest for the baseline, moderate for Ridge and largest for the tree ensembles. This is the pattern the research question asks about, with the caveat that it rests on one model per family and a few variants.
- A larger gap does not mean a worse model: under chronological validation the forest is still the best of the models. What changes is the *size of its advantage*. Relative to the hour x day type baseline, random validation shows the forest 41% better and Ridge 20% better, whereas chronological validation shows 16% and 8% (boosting: **[fill in]**). Random validation therefore overstates the benefit of a flexible model, and the difference between Ridge variants can flip with the scheme (see the `ridge` and `ridge_log` rows of the RMSE table).

### 3. What blocked validation shows
- Blocked validation is close to chronological validation for most models, so most of the optimism comes from neighbouring hours and not from the direction of time. The remaining difference between blocked and chronological also reflects different training sets (blocked trains on all other blocks, chronological only on earlier data and on less of it in the early folds), so it should not be attributed to the direction of time alone.
- Blocked validation is not a forecasting simulation, because its training data can lie after the test block. It nevertheless shows a mean residual of about the same size as chronological validation (+74 to +90 for Ridge, Ridge with a log target and the forest; check that boosting falls in the same range), so in this dataset it did not hide the underestimation of 2012.

### 4. Main failure mode: the level shift between the years
- Every model, including both baselines and Ridge, underestimates demand in 2012 in every month by a similar amount (Section 4.3; chronological bias about +73 to +87 rentals per hour for the models listed in the bias table; check that boosting falls in the same range). Because the linear and the tree models behave alike, the dominant cause is the growth in demand from 2011 to 2012 that no feature set represents (no `yr`, no lagged demand), not the bounded predictions of trees. The bounding of tree predictions can still add to the error at the very highest demand; this was not tested here.
- Even under random validation the 2012 hours are underestimated on average (bias +31 to +53), which is consistent with training data that mixes the lower 2011 level with the higher 2012 level without year information.
- Errors are largest in working-day commuting peaks (Section 4.4); **[fill in the peak and off-peak numbers]**.

### 5. Limitations
- Two years of data from one city; the conclusions are about this dataset and these model families, not about hourly demand forecasting in general.
- Random validation was run once with one seed, and the bootstrap intervals cover only the choice of test hours, not the randomness of training or of the hyperparameter search.
- Remedies such as `yr`, lagged demand or de-trending were not evaluated here. The log target did not reduce the bias of Ridge (Section 2 table), so it is not a tested remedy for the level shift.
- The hyperparameter grids are coarse, and some selected values lie on the edge of the grid (see the model notebooks).